In [ ]:
import xarray as xr

In [ ]:
from matplotlib import pyplot as plt

## SCHISM-WindWaveNet Integration Test

Test with different architectures, significant wave height initalised from constant value (1.3 m, training data mean) -- spinup from 0 does not create any significant wave height (or very slowly).

This test case is by no means physically realistic and only serves to see whether coupling works (WindWaveNet is called and something is happening), and to get runtime estimates

In [ ]:
#ds_twoway = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WWM_Duck/outputs-original/out2d_*.nc")
ds_oneway = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WWM_Duck/outputs-elfe-2/out2d_*.nc")
ds_wwnet_mlp = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WindWaveNet_MLP_Duck/outputs/out2d_*.nc")
ds_wwnet_lstm = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WindWaveNet_LSTM_Duck/outputs/out2d_*.nc")
ds_wwnet_graph = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WindWaveNet_Graph_Duck/outputs/out2d_*.nc")
ds_wwnet_graph_vn = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WindWaveNet_Graph-VN_Duck/outputs/out2d_*.nc")
ds_wwnet_graph_lstm_vn = xr.open_mfdataset("/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/Test_WindWaveNet_Graph-LSTM-VN_Duck/outputs/out2d_*.nc")

In [ ]:
ds = xr.concat([
    #ds_twoway, 
    ds_oneway, 
    ds_wwnet_mlp, 
    ds_wwnet_lstm, 
    ds_wwnet_graph,
    ds_wwnet_graph_vn,
    ds_wwnet_graph_lstm_vn,
], dim="experiment")

ds = ds.assign_coords({"experiment":[
    #"SCHISM-WWM-both-ways", 
    "SCHISM-WWM-one-way", 
    "SCHISM-WindWaveNet-MLP", 
    "SCHISM-WindWaveNet-LSTM", 
    "SCHISM-WindWaveNet-Graph",
    "SCHISM-WindWaveNet-Graph-VN",
    "SCHISM-WindWaveNet-Graph-LSTM-VN",
]})

In [ ]:
ds

This should look the same for all experiments

In [ ]:
ds["elevation"].mean(dim="nSCHISM_hgrid_node").plot()

This should not look the same for all experiments

In [ ]:
ds["sigWaveHeight"].mean(dim="nSCHISM_hgrid_node").plot()

In [ ]:
for i in list(range(ds.sizes["experiment"])):
    ds.isel(experiment=i)["sigWaveHeight"].mean(dim="nSCHISM_hgrid_node").plot(label=ds.isel(experiment=i).experiment.values)

plt.legend()
plt.title("")

plt.show()

In [ ]:
max_swh = ds["sigWaveHeight"].max(dim=("time", "nSCHISM_hgrid_node")).values
plt.plot(max_swh, list(range(len(ds.experiment))), "o")
plt.gca().set_yticks(list(range(len(ds.experiment))))
plt.gca().set_yticklabels(ds.experiment.values);
plt.gca().grid()

In [ ]:
#ds.isel(experiment=2)["sigWaveHeight"].max(dim="nSCHISM_hgrid_node").plot()
ds.isel(experiment=3)["sigWaveHeight"].max(dim="nSCHISM_hgrid_node").plot()

In [ ]:
for i in list(range(ds.sizes["experiment"])):
    ds.isel(experiment=i)["elevation"].mean(dim="nSCHISM_hgrid_node").plot(label=ds.isel(experiment=i).experiment.values)

plt.legend()
plt.title("")

plt.show()

In [ ]:
fig, axs = plt.subplots(len(ds.experiment), 4, sharex=True, sharey=True, figsize=(20, 4*len(ds.experiment)))

for j in range(len(ds.experiment)):
    axj = axs[j]
    for i, ax in zip([0, 15, 30, 45], axj):
        img = ax.scatter(ds.SCHISM_hgrid_node_x.values, ds.SCHISM_hgrid_node_y.values, c=ds.isel(time=i, experiment=j).sigWaveHeight.values, alpha=1.0, vmin=0)
        plt.colorbar(img, ax=ax)
        ax.set_title(f"{ds.experiment.values[j]}: {str(ds.time.values[i])[:16]}")

fig.tight_layout()

## Runtime from logs

In [ ]:
import re
import numpy as np

In [ ]:
experiments = [
    "Test_WWM_Duck",
    "Test_WindWaveNet_MLP_Duck",
    "Test_WindWaveNet_LSTM_Duck",
    "Test_WindWaveNet_LSTM-48_Duck",
    "Test_WindWaveNet_Graph_Duck",
    "Test_WindWaveNet_Graph-VN_Duck",
    "Test_WindWaveNet_Graph-LSTM-VN_Duck",
    "Test_WindWaveNet_Graph-LSTM-VN-48_Duck",
]

In [ ]:
mean_walltimes = []
std_walltimes = []

In [ ]:
skip_lines = 5 # libtorch spinup

In [ ]:
for exp in experiments:
    logfile = f"/work/ka1176/caroline/gitlab/schism-windwavenet/schism-experiments/{exp}/outputs/mirror.out"

    matching_lines = []

    with open(logfile, "r") as f:
        lines = f.readlines()
        lines = [ll for ll in lines if "took" in ll]

        walltimes = [float(re.findall(r"((\d.\d+)(E-\d+)?)", ll)[0][0]) for ll in lines[skip_lines:]]

    #print(lines[-1], walltimes[-1])

    print(f"{exp:25s}: mean = {np.array(walltimes).mean()*1e3:.4f} ms, std={np.array(walltimes).std()*1e3:.4f} ms")
    mean_walltimes.append(np.array(walltimes).mean()*1e3)
    std_walltimes.append(np.array(walltimes).std()*1e3)

In [ ]:
best_rmse = [0,
             0.0502,
             0.0208,
             0.0189,
             0.0559,
             0.0429,
             0.0247,
             0.0249
            ]

In [ ]:
mem_domain_mb = [1.8, 2.014, 0.448, 1.624, 0.39, 2.21, 2.45, 2.45]

In [ ]:
import pandas as pd
import seaborn as sns
from matplotlib import pyplot as plt

In [ ]:
df = pd.DataFrame([experiments, mean_walltimes, std_walltimes, best_rmse, mem_domain_mb]).T
df.columns = ["experiment", "wall_time_ms", "wall_time_std_ms", "swh_rmse_m", "mem_domain_mb"]

In [ ]:
df.to_csv("/work/ka1176/caroline/gitlab/wind-wave-model-emulator/notebooks/paper-plots/performance_metrics.csv")

In [ ]:
fig, ax = plt.subplots(1, 1, figsize=(3.5, 2.6))
sns.scatterplot(data=df, x="wall_time_ms", y="swh_rmse_m", size="mem_domain_mb")